# 例子和测试: 摆锤环境使用 sac 算法训练智能体

## import

In [6]:
import matplotlib.pyplot as plt
from torch import Tensor, concat
from torch.nn import Linear, ReLU, Sequential
from torch.nn.functional import softplus
from torch.nn.init import xavier_uniform_, zeros_

from modelsolver import AgentModelSolver
from modelsolver.abc.config import (AgentConfig, HyperParameterConfig,
                                    ReplayBufferConfig)
from modelsolver.abc.model import IActor, ICritic
from modelsolver.implement.environment.pendulum import (PendulumConfig,
                                                        PendulumEnvironment)

## 定义智能体模型

In [7]:
class SimpleActor(IActor):
    def __init__(self, config: AgentConfig):
        super().__init__()
        self._config = config
        self.sequential = self._create_policy_network()
        self.mean = Linear(self.config.hidden_channels, self.config.action_channels)
        self.std = Linear(self.config.hidden_channels, self.config.action_channels)
        self._init_params()

    @property
    def config(self) -> AgentConfig:
        return self._config  # type: ignoree

    def _forward(self, state: Tensor) -> Tensor:
        return self.sequential(state)

    def _action_mean(self, x: Tensor) -> Tensor:
        return self.mean(x)

    def _action_std(self, x: Tensor) -> Tensor:
        return softplus(self.std(x))

    def _create_policy_network(self):

        return Sequential(
            Linear(self.config.state_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, self.config.hidden_channels),
            ReLU(),
        )

    def _init_params(self):
        for module in self.children():
            if isinstance(module, Linear):
                xavier_uniform_(module.weight)
                zeros_(module.bias)


class SimpleCritic(ICritic):
    def __init__(self, config: AgentConfig):
        super().__init__()
        self._config = config
        self.sequential = self._create_value_network()
        self._init_params()

    @property
    def config(self) -> AgentConfig:
        return self._config  # type: ignore

    def forward(self, states, actions):
        return self.sequential(concat([states, actions], dim=-1))

    def _create_value_network(self):

        return Sequential(
            Linear(self.config.state_channels + self.config.action_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, 1)
        )

    def _init_params(self):
        for module in self.children():
            if isinstance(module, Linear):
                xavier_uniform_(module.weight)
                zeros_(module.bias)

## 设置配置项

In [ ]:
S = 3
A = 1
agent_config = AgentConfig(
    state_channels=S,
    action_channels=A,
    hidden_channels=128,
    target_entropy=-A
)
# sac: c:3e-2, a:3e-2 gamma_tl:0.98
hyper_params_config = HyperParameterConfig(learning_rate=3e-4,
                                           milestones=[999],
                                           gamma_rl=0.98,
                                           critic_lr=3e-2,
                                           actor_lr=3e-2,
                                           epoch=300,
                                             policy_delay=2)

replay_buffer_config = ReplayBufferConfig(
    capacity=100000,
    state_dim=S,
    action_dim=A,
    minimal_capacity=1024,
    batch_size=512
)
env_config = PendulumConfig(
    terminated_delta=5,
    truncated_time=50,
)

## 构建解决方案

In [9]:
solver = AgentModelSolver()\
    .add_actor_component(SimpleActor)\
    .add_critic_component(SimpleCritic)\
    .add_model_config(agent_config)\
    .add_environment(PendulumEnvironment)\
    .add_environment_config(env_config)\
    .add_config(hyper_params_config)\
    .add_config(replay_buffer_config)

## 训练过程

In [ ]:
solver.train(0, "sac")

solver.train(0, "td3")

NameError: name 'solver' is not defined

: 